# CSE598/EEE589 Project 05 — setup and preliminary checks

**Long Sequence Attention Across GPUs.** This notebook prepares a fresh runtime, runs the CPU correctness tests, shows the analytical causal-imbalance model, and classifies the experiment matrix. It does not measure distributed throughput.

## Run this on a fresh Google Colab runtime

1. On your computer, from the project root, build the upload package. It excludes the virtualenv, the CPU lockfile, and internal instruction files:

```bash
python3 scripts/make_colab_package.py
```

2. Open Google Colab. **File → Upload notebook** and choose `notebooks/01_colab_setup_and_preliminary.ipynb` from the package, or upload that notebook by itself.
3. **Runtime → Change runtime type → GPU → Save.** The assigned GPU may be a T4, L4, A100, or another device. Do not assume which one you will get.
4. Upload `spbench_colab.zip` so it lands at `/content/spbench_colab.zip`.
5. In the configuration cell, set `ZIP_PATH = "/content/spbench_colab.zip"`. Leave `REPO_URL` empty unless you clone a real commit.
6. **Runtime → Run all.**
7. Download the ZIP the last cell writes before the runtime is recycled.

The installer keeps Colab's existing PyTorch. It does not install `requirements-cpu-dev.txt`.


## Configuration


In [ ]:
# Fresh Colab runtime: upload the project ZIP to /content and set ZIP_PATH.
# If the repository has a remote, set REPO_URL and the full REPO_COMMIT instead.
# Leave all three empty when this notebook is already running inside the project tree.
REPO_URL = ""
REPO_COMMIT = ""
ZIP_PATH = ""
PROJECT_DIR = ""


## Repository bootstrap


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

def discover(start):
    root = Path(start).resolve()
    pool = [root, *root.parents]
    if root.is_dir():
        for child in root.iterdir():
            if child.is_dir() and child.name not in {".venv", ".git"}:
                pool.append(child)
                for grandchild in child.iterdir():
                    if grandchild.is_dir() and grandchild.name not in {".venv", ".git"}:
                        pool.append(grandchild)
    for candidate in pool:
        if (candidate / "src" / "spbench" / "__init__.py").is_file():
            return candidate
    return None

if ZIP_PATH:
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_unzipped")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(dest)
    project = discover(dest)
elif REPO_URL:
    if not REPO_COMMIT:
        raise ValueError("Set REPO_COMMIT to a full commit hash. Do not track a moving branch.")
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_clone")
    if not (dest / ".git").exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    subprocess.check_call(["git", "checkout", REPO_COMMIT], cwd=dest)
    project = discover(dest)
else:
    project = Path(PROJECT_DIR).resolve() if PROJECT_DIR else discover(Path.cwd())

if project is None:
    raise FileNotFoundError(
        "No project source found. On a fresh Colab runtime, upload the project ZIP and set ZIP_PATH, "
        "or set REPO_URL and REPO_COMMIT."
    )
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
print("Project directory:", project)


## Install the project without upgrading an existing PyTorch


In [ ]:
import importlib.util
bootstrap_path = project / "scripts" / "colab_bootstrap.py"
spec = importlib.util.spec_from_file_location("colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
bootstrap.install_project(project)
import spbench
print("spbench", spbench.__version__)


## Runtime and dependency preflight


In [ ]:
import json
from spbench.dependencies import probe_dependencies
from spbench.hardware import detect_hardware

hardware = detect_hardware()
dependencies = probe_dependencies()
print(json.dumps({"hardware": hardware, "dependencies": dependencies}, indent=2, default=str))
if hardware["mps_available"] and not hardware["cuda_available"]:
    print("Apple MPS is visible. It is not used as an NVIDIA baseline.")


## CPU reference and partition tests


In [ ]:
import subprocess
import sys
completed = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=project, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError("CPU tests failed. Distributed timing was not started.")
print("CPU tests passed. This does not establish GPU or NCCL correctness.")


## Analytical causal imbalance


In [ ]:
from spbench.demo import write_causal_imbalance_figure
summary = write_causal_imbalance_figure(project / "results" / "figures" / "analytical" / "causal_imbalance_s1024_p8.png")
print(summary["weak_context_note"])
for layout, ratios in summary["imbalance"].items():
    print(layout, {key: round(value, 4) for key, value in ratios.items()})
print("Figure:", summary["path"])
print("These pair counts are formulas, not kernel times.")


## Dry-run experiment matrix


In [ ]:
from spbench.sweep import main as sweep_main
code = sweep_main([
    "--config", str(project / "configs" / "benchmark.yaml"),
    "--dry-run",
    "--include-hypothetical-plan",
    "--output", str(project / "results" / "processed" / "dry_run_matrix.json"),
])
if code != 0:
    raise RuntimeError(f"dry-run exited {code}")
print("Dry-run classification finished. Rejected rows are illegal configurations. Blocked rows were not executed.")


## Export the preliminary evidence


In [ ]:
import zipfile
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archive_path = project / "results" / f"phase1_evidence_{stamp}.zip"
include = [
    project / "reports" / "preliminary_findings.md",
    project / "STATUS.md",
    project / "results" / "processed" / "dry_run_matrix.json",
    project / "results" / "figures" / "analytical" / "causal_imbalance_s1024_p8.png",
    project / "configs" / "benchmark.yaml",
    project / "docs" / "upstream_audit.md",
]
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in include:
        if path.is_file():
            archive.write(path, path.relative_to(project))
        else:
            print("missing, not packaged:", path)
print("Evidence archive:", archive_path)
print("Download this file from the Colab file browser, or copy it out of results/ locally.")
